# Document SHAP values

Compute token attributions for each saved test fold, then combine the fold tables.

In [ ]:
from shared import paths
from IPython.display import display


## Setup

In [ ]:
import pandas as pd
import glob
import shap
import torch
from transformers import pipeline, AutoTokenizer
import pickle
import numpy as np
import os
from shared.shap_values import truncate_long_texts, merge_shap_values_with_offsets

paths.prepare_directories()
CV_BASE_PATH = str(paths.document_run())
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')


## Fold attributions

In [ ]:
for fold in range(1, 11):
    print(f'\n Processing fold {fold}...')
    shap_path = os.path.join(CV_BASE_PATH, f'cv_fold_{fold}/logs/shap_data_fold_{fold}.pkl')
    if not os.path.exists(shap_path):
        print(f' Missing SHAP input for fold {fold}: {shap_path}')
        continue
    with open(shap_path, 'rb') as f:
        shap_data = pickle.load(f)
    model_path = paths.resolve_checkpoint(shap_data['model_path'], paths.document_run() / f'cv_fold_{fold}')
    test_data = shap_data['tokenized_test_data']
    test_texts = test_data['raw_texts']
    test_ids = test_data['letter_ids']
    test_labels = test_data['true_labels']
    print(f' Model: {model_path}')
    print(f' Test instances: {len(test_texts)}')
    tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')
    model = pipeline(
        'text-classification',
        model=model_path,
        tokenizer=tokenizer,
        device=0 if device.type != 'cpu' else -1,
        top_k=None,
    )
    test_texts_trunc = truncate_long_texts(test_texts, test_ids, tokenizer)
    print(f' Computing SHAP values for fold {fold}...')
    explainer = shap.Explainer(model, tokenizer)
    shap_values = explainer(test_texts_trunc)
    final_dfs = []
    for i, (text, shap_vals, text_id) in enumerate(zip(test_texts, shap_values.values, test_ids)):
        merged_tokens, merged_shap = merge_shap_values_with_offsets(text, shap_vals, tokenizer)
        shap_df = pd.DataFrame(merged_shap, columns=['shap_noRef', 'shap_Ref'])
        shap_df.insert(0, 'token', merged_tokens)
        shap_df.insert(0, 'text_id', text_id)
        final_dfs.append(shap_df)
    merged_shap_df = pd.concat(final_dfs, ignore_index=True)
    print('\n--- Final Merged Tokens & SHAP Values ---')
    print('[LETTER TEXT]')
    output_path = os.path.join(CV_BASE_PATH, f'cv_fold_{fold}/logs/shap_output_fold_{fold}.csv')
    merged_shap_df.to_csv(output_path, index=False)
    print(f' SHAP results for fold {fold} saved to: {output_path}')


## Combined table

In [ ]:
import pandas as pd
import glob
import os

base_dir = CV_BASE_PATH
output_dir = os.path.join(base_dir, 'general')
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'shap_output_all_folds.csv')
fold_paths = glob.glob(os.path.join(base_dir, 'cv_fold_*/logs/shap_output_fold_*.csv'))
dfs = []
for path in fold_paths:
    try:
        fold_number = int(path.split('cv_fold_')[1].split('/')[0])
    except Exception:
        print(f' Could not read the fold number from: {path}')
        continue
    df = pd.read_csv(path)
    df['fold'] = fold_number
    dfs.append(df)
if dfs:
    combined_df = pd.concat(dfs, ignore_index=True)
    combined_df.to_csv(output_path, index=False)
    print(f' Combined SHAP table saved to:\n{output_path}')
else:
    print(' No SHAP output files found.')
